In [ ]:
from pysidt import Datum, Node, SubgraphIsomorphicDecisionTree
from molecule.molecule import Molecule,Group
from molecule.molecule.atomtype import ATOMTYPES
import numpy as np
import json
import pickle
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
import json
with open("../data/RMG_Habs_low_rank_adjlist_toAnEa_train.json",'r') as f:
    train_json = json.load(f)
with open("../data//RMG_Habs_low_rank_adjlist_toAnEa_val.json",'r') as f:
    val_json = json.load(f)   
with open("../data/RMG_Habs_low_rank_adjlist_toAnEa_test.json",'r') as f:
    test_json = json.load(f)

In [ ]:
#Extract and compute Ln(k(1000K))
train = [Datum(Molecule().from_adjacency_list(adj_AnE[0],check_consistency=False),value=adj_AnE[1][0]+adj_AnE[1][1]*np.log(1000.0)-adj_AnE[1][2]/(8.314*1000.0)) for adj_AnE in train_json]
val = [Datum(Molecule().from_adjacency_list(adj_AnE[0],check_consistency=False),value=adj_AnE[1][0]+adj_AnE[1][1]*np.log(1000.0)-adj_AnE[1][2]/(8.314*1000.0)) for adj_AnE in val_json]
test = [Datum(Molecule().from_adjacency_list(adj_AnE[0],check_consistency=False),value=adj_AnE[1][0]+adj_AnE[1][1]*np.log(1000.0)-adj_AnE[1][2]/(8.314*1000.0)) for adj_AnE in test_json]

In [ ]:
tree = SubgraphIsomorphicDecisionTree(
        root_group = Group().from_adjacency_list("""
1 *1 R u[0,1,2,3] px cx {2,S}
2 *2 H u0 p0 c0 {1,S}
3 *3 R u[0,1,2,3] px cx
"""),
        r=[ATOMTYPES[x] for x in ["C", "O", "N", "S", "H"]],
        r_bonds=[1, 2, 3, 1.5],
        r_un=[0,1,2,3],
        max_nodes=200,
        iter_max=2,
        iter_item_cap=100,
        max_structures_to_generate_extensions=1000,
    )

In [ ]:
tree.generate_tree(data=train,validation_set=None,nprocs=4,checkpoint_every=50)

In [ ]:
tree.regularize()

In [ ]:
tree.prune(val,N=100)

In [ ]:
test_pred = np.array([np.exp(tree.evaluate(d.mol)) for d in test])
test_value = np.array([np.exp(d.value) for d in test])

In [ ]:
test_log_error = np.log(test_pred)-np.log(test_value)

In [ ]:
np.mean(np.abs(test_log_error))

In [ ]:
_,bins = np.histogram(test_log_error)
plt.hist(test_pred/test_value,np.exp(bins))
plt.xscale("log")
plt.xlabel("kpred(1000 K)/ktrue(1000 K)")
plt.ylabel("Counts")

In [ ]:
#Restart from checkpoint
with open("sidt_checkpoint_101_nodes.pkl",'rb') as f:
    node_dict = pickle.load(f)
    nodes = {name: Node(group=d["group"],rule=d["rule"],parent=d["parent"],children=d["children"],name=name,depth=d["depth"]) for name,d in node_dict.items()}

for node in nodes.values():
    if node.parent:
        node.parent = nodes[node.parent]
    else:
        node.parent = None
    node.children = [nodes[child] for child in node.children]
    

In [ ]:
tree_restart = SubgraphIsomorphicDecisionTree(
        nodes=nodes,
        r=[ATOMTYPES[x] for x in ["C", "O", "N", "S", "H"]],
        r_bonds=[1, 2, 3, 1.5],
        r_un=[0,1,2,3],
        max_nodes=200,
        iter_max=2,
        iter_item_cap=100,
        max_structures_to_generate_extensions=1000,
    )

In [ ]:
tree_restart.generate_tree(data=train,validation_set=None,nprocs=4,checkpoint_every=None)
tree_restart.regularize()
tree_restart.prune(val,N=100)

In [ ]:
test_pred = np.array([np.exp(tree_restart.evaluate(d.mol)) for d in test])
test_value = np.array([np.exp(d.value) for d in test])
test_log_error = np.log(test_pred)-np.log(test_value)
np.mean(np.abs(test_log_error))

In [ ]:
_,bins = np.histogram(test_log_error)
plt.hist(test_pred/test_value,np.exp(bins))
plt.xscale("log")
plt.xlabel("kpred(1000 K)/ktrue(1000 K)")
plt.ylabel("Counts")